# 04 · Enrichment, 500-token windows and embeddings

Plan sections 3.3 and 3.4; pipeline steps 1–2. Everything here runs on the CPU (vLLM is not needed).

* **Enrichment** (deterministic, `csrag.enrich`):
  * `engine_type` from the subpart: A → `all`, B/C → `piston`, D/E/F → `turbine`;
  * character spans (`char_start`/`char_end`) of every base unit in its section's whitespace-normalised `body`, the same span fields the windows and the gold quotes have;
  * `cross_refs`: `CS-E …` and `AMC E …` / `AMC to CS-E …` references, normalised to section ids that exist, in order of mention.
* **Windows** (`csrag.windows`): at most 500 bge-m3 tokens, consecutive windows share ≥ 50 tokens, windows never cross a section, and a section of ≤ 500 tokens is exactly one window.
* **Embeddings**: one bge-m3 forward pass in the encoder container (on the GPU in phase A) gives the dense, learned-sparse and ColBERT vectors; BM25 comes from fastembed with `avg_len` set per index.

**Writes:** `data/windows_500.jsonl`, `data/cross_refs.jsonl`, `data/embeddings/<collection>.{npz,jsonl}`.

**Phase A (GPU encoder).** Notebooks 04–07 run with the encoder on the GPU and **vLLM stopped**; notebook 08 swaps to vLLM.
```bash
docker compose stop vllm encoder                   # the GPU encoder and vLLM do not fit together
docker compose up -d qdrant && docker compose up -d --force-recreate encoder-gpu
```
**Experiments.** Notebooks 04 and 05 build the corpus side (windows, embeddings, Qdrant collections). Every experiment shares it, so nothing here depends on the experiment and nothing is written to `results/`. The experiment, i.e. the name of the results folder `results/<name>/` and the question files that are run, is set in **`experiments.toml`** at the repo root: `active = "<name>"`, or `CSRAG_EXPERIMENT=<name>` in the shell or `.env` for one session. Notebooks 06–09 read it and print it in their first cell. `PARAMS_VERSION` (notebooks 07 and 08) is a different setting: it picks the parameter set, not the folder, and one experiment holds the v1 and the v2 runs.

Embedding both indexes (~155k + ~165k tokens) takes a few minutes on the GPU (15–40 min on the CPU `encoder`, which also works). Vectors are cached by text hash, so a re-run only embeds texts that changed.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"

In [2]:
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from csparse.merge import read_jsonl, write_jsonl
from csrag.corpus import COLLECTIONS, UNIT_FILES, Corpus, bge_tokenizer, count_tokens
from csrag.enrich import corpus_version, cross_ref_records, unit_spans
from csrag.windows import check_windows, make_windows, window_stats

tok = bge_tokenizer()
sections = read_jsonl(DATA / "sections.jsonl")
sec = {s["section_id"]: s for s in sections}
groups = read_jsonl(DATA / "groups.jsonl")
chunks = read_jsonl(DATA / "chunks.jsonl")
print(len(sections), "sections,", len(groups), "groups,", len(chunks), "base units")

175 sections, 92 groups, 252 base units


## Inputs (plan section 0)

bge-m3 tokens of each record's `text` (no special tokens). 8192 is bge-m3's input limit.

In [3]:
def token_stats(texts):
    n = np.array([count_tokens(t) for t in texts])
    return {"count": len(n), "median": int(np.median(n)), "p90": int(np.percentile(n, 90)),
            "max": int(n.max()), "> 8192": int((n > 8192).sum()), "total": int(n.sum())}

inputs = pd.DataFrame({
    "base units (chunks.jsonl)": token_stats(c["text"] for c in chunks),
    "sections (sections.jsonl)": token_stats(s["text"] for s in sections),
    "groups (groups.jsonl)": token_stats(g["text"] for g in groups),
}).T
inputs

,count,median,p90,max,> 8192,total
base units (chunks.jsonl),252,463,1414,1817,0,155117
sections (sections.jsonl),175,309,1846,11232,3,153170
groups (groups.jsonl),92,470,4357,17923,5,153170


## Windows

Rule (plan 3.3): a window ends at the last line or sentence end within its final 100 tokens (or at a word end if there is none); the next window starts 50 tokens before the previous one ends, moved back to the start of a word. `text` = the section's header line + the window body; the header is not counted in the 500.

**Checks (every list must be empty):** windows cover each section's whole body, no window crosses a section, every window body has ≤ 500 tokens, consecutive windows share ≥ 50 tokens, and every section of ≤ 500 tokens is exactly one window.

In [4]:
windows = make_windows(sections, tok, size=500, overlap=50, tail=100)
problems = check_windows(windows, sec, tok, size=500, overlap=50)
for k, v in problems.items():
    print(f"{k:28} {len(v):3}  {v[:5]}")
assert not any(problems.values()), "window checks failed"
write_jsonl(windows, DATA / "windows_500.jsonl")
pd.Series(window_stats(windows))

sections_without_windows       0  []
not_covered                    0  []
span_text_mismatch             0  []
too_long                       0  []
overlap_too_small              0  []
short_section_not_single       0  []


windows                      436.0
sections                     175.0
single_window_sections       116.0
boundaries                   261.0
tokens_median                465.5
tokens_p90                   496.0
tokens_max                   500.0
tokens_total              164115.0
dtype: float64

The plan's prototype reported ~486 windows and ~189k window tokens. The rule as written gives fewer, because its overlaps are close to the 50-token minimum (see the overlap distribution below). The numbers that matter for RQ7 (window size ≤ 500, overlap ≥ 50, section-bounded) are checked above.

In [5]:
from csrag.enrich import normalize_ws
overlaps = []
for a, b in zip(windows, windows[1:]):
    if a["section_id"] == b["section_id"]:
        nb_ = normalize_ws(sec[a["section_id"]]["body"])
        overlaps.append(count_tokens(nb_[b["char_start"]:a["char_end"]]))
per_section = pd.Series([w["section_id"] for w in windows]).value_counts()
print("window boundaries:", len(overlaps), "| overlap tokens min/median/max:", min(overlaps), np.median(overlaps), max(overlaps))
print("sections with one window:", int((per_section == 1).sum()), "of", len(per_section))
per_section.head(10).rename("windows").to_frame()

window boundaries: 261 | overlap tokens min/median/max: 50 50.0 57
sections with one window: 116 of 175


,windows
AMC E 780,26
CS-E 740,22
AMC E 515,20
AMC E 650,17
AMC E 790(a)(2),17
AMC E 740(c)(4),15
AMC E 510,13
AMC E 1030,12
AMC E 130,11
CS-E 800,11


## Spans of the base units

Each unit's body is located in its section's whitespace-normalised body. A unit that starts with a repeated table header (from the 1500-token split) is not a contiguous substring; it is located by its tail (`span_exact = False`).

In [6]:
spans = unit_spans(chunks, sec)
inexact = [k for k, v in spans.items() if not v["span_exact"]]
print(len(spans), "unit spans found;", len(inexact), "located by their tail:", inexact)
assert len(spans) == len(chunks)

252 unit spans found; 0 located by their tail: []


## Cross-references

Extracted per unit and per window, in order of mention, self-references dropped. `data/cross_refs.jsonl` has one row per (index, source unit, referenced section).

In [7]:
corpus = Corpus(DATA)  # now also loads windows_500.jsonl
rows = []
for index in corpus.indexes():
    rows += cross_ref_records(corpus.units(index).values(), index, corpus.section_ids)
write_jsonl(rows, DATA / "cross_refs.jsonl")
corpus = Corpus(DATA)  # reload: cross_refs now come from the file

refs = pd.DataFrame(rows)
summary = refs.groupby("index").agg(refs=("dst_section_id", "size"), units_with_refs=("src_unit_id", "nunique"),
                                    sections_referenced=("dst_section_id", "nunique"))
summary["units"] = [len(corpus.units(i)) for i in summary.index]
display(summary)
refs[refs["index"] == "units"]["dst_section_id"].value_counts().head(10).rename("references (units index)").to_frame()

,refs,units_with_refs,sections_referenced,units
index,,,,
units,446,189,141,252
win500,531,283,141,436


,references (units index)
dst_section_id,
CS-E 740,33
CS-E 20,24
CS-E 510,23
CS-E 50,13
CS-E 515,12
CS-E 30,12
CS-E 25,12
CS-E 440,11
CS-E 780,11


In [8]:
pd.DataFrame({i: pd.Series([u["engine_type"] for u in corpus.units(i).values()]).value_counts() for i in corpus.indexes()})

,units,win500
turbine,162,299
all,56,98
piston,34,39


In [9]:
versions = {i: corpus_version(DATA / UNIT_FILES[i]) for i in corpus.indexes()}
versions

{'units': 'sha256:7a8a726c1198e965fc1467dcbf59dfcacac17309bcb94e8f293ebe70a0b5ce46',
 'win500': 'sha256:bcb1dac980c90cf72a5f5d50ac7d6d3edc84f5cf50098b8586991fb238507031'}

## Embeddings (encoder container, GPU in phase A)

* bge-m3 through `POST /embed`: dense (1024-d, normalised), learned sparse (lexical weights) and ColBERT (1024-d per token), all from one forward pass, `max_length` 8192, fp32.
* The embedded text is the unit or window `text` (header line + body), with no instruction prefix.
* BM25: fastembed `Qdrant/bm25`, k1 = 1.2, b = 0.75 and `avg_len` = the index's mean BM25 document length; Qdrant applies the IDF (`modifier=IDF`).

In [10]:
from csrag.embed import BM25, EmbeddingStore, EncoderClient, bm25_avg_len
from csrag.retrievers.base import PARAMS

enc = EncoderClient()
health = enc.wait_ready()
print("encoder on", health["device"], health.get("gpu") or "", "| dtype", health["dtype"])
pd.DataFrame(health["models"]).T[["repo", "revision", "dtype", "device"]]

encoder on cuda NVIDIA GeForce RTX 3090 | dtype float32


,repo,revision,dtype,device
bge_m3,BAAI/bge-m3,5617a9f61b028005a4858fdac845db406aefb181,float32,cuda
bge_reranker,BAAI/bge-reranker-v2-m3,953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e,float32,cuda
jina_reranker,jinaai/jina-reranker-v3,d7d7e73b6ea138ced340b83865931b5dfb6c97aa,float32,cuda


In [11]:
bm25_params = PARAMS["v1"]["bm25"]
build = {}
for index in corpus.indexes():
    units = list(corpus.units(index).values())
    avg_len = bm25_avg_len(u["text"] for u in units)
    store = EmbeddingStore(COLLECTIONS[index], DATA / "embeddings")
    build[index] = {"avg_len": round(avg_len, 2), **store.build(units, enc, BM25(avg_len=avg_len, **bm25_params),
                                                                 corpus.n_tokens, progress=tqdm)}
pd.DataFrame(build).T

embed cse_units_v1:   0%|          | 0/25 [00:00<?, ?it/s]

embed cse_win500_v1:   0%|          | 0/28 [00:00<?, ?it/s]

,avg_len,embedded,bm25_recomputed,encoder_seconds
units,262.17,252,True,28.5
win500,168.33,436,True,24.2


### Embedding checks

Dense vectors are unit length; each unit has one ColBERT vector per token after `[CLS]`. Qdrant holds at most 1023 ColBERT vectors per point (a multivector must have fewer than 1,048,576 numbers): longer base units are stored in Qdrant without their ColBERT vector, and C1/M1 score ColBERT MaxSim client-side over the full cached vectors for both indexes (`colbert_engine = "client"`; notebook 05 checks that the two engines agree).

In [12]:
from csrag.index import MAX_MULTIVECTOR
rows = []
for index in corpus.indexes():
    store = EmbeddingStore(COLLECTIONS[index], DATA / "embeddings")
    units = corpus.units(index)
    norms = np.array([np.linalg.norm(store.get(u)["dense"]) for u in units])
    ncol = np.array([len(store.get(u)["colbert"]) for u in units])
    ntok = np.array([corpus.n_tokens(units[u]["text"]) + 1 for u in units])
    rows.append({"index": index, "units": len(units), "missing": len(store.missing(units.values())),
                 "dense_norm_min": norms.min(), "dense_norm_max": norms.max(),
                 "colbert_rows_match_tokens": float((ncol == ntok).mean()), "colbert_vectors": int(ncol.sum()),
                 f"units_over_{MAX_MULTIVECTOR}_vectors": int((ncol > MAX_MULTIVECTOR).sum()),
                 "npz_MB": round(store.npz.stat().st_size / 2**20, 1)})
pd.DataFrame(rows).set_index("index")

,units,missing,dense_norm_min,dense_norm_max,colbert_rows_match_tokens,colbert_vectors,units_over_1023_vectors,npz_MB
index,,,,,,,,
units,252,0,1.0,1.0,1.0,155369,60,305.4
win500,436,0,1.0,1.0,1.0,171437,0,337.7
